In [ ]:
from IPython.display import clear_output
!pip install catboost
clear_output()

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split, KFold
from catboost import CatBoostRegressor
import numpy as np

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

# We basically read the path of the csv path with the respected csv name, then we read the data and store it into df -> Pandas DataFrame format for csv files
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:

# Here we simply just use the .head() function to read the first 5 rows of the Dataset
df.head()

In [ ]:
# Task 3: Write your code here:

# Here we use the info() function to give us a clear summary of the dataset
df.info()

In [ ]:
# Task 4: Write your code here:

# Here we show the statistical insights of the given dataset

df.describe()

In [ ]:
# Task 5: Write your code here:

# Here we first extract the values of the Target variable (Delivery_Time) and then we plot the distribution of it with a proper number of bins

deliv_time = df['Delivery_Time']

plt.figure(figsize=(10, 5))
plt.hist(deliv_time, bins=20, edgecolor='black')
plt.xlabel("Values")
plt.ylabel("Frequency")
plt.show()

# INSIGHTS!
# As we can see from the distribution of the data, it's following a Normal Distribution or a Gaussian Distribution which is generally good :D

In [ ]:
# Task 1: Write your code here:

# Here we basically drop the Order_ID column and show the first 5 rows as a confirmation for the deletion

df = df.drop("Order_ID", axis = 1)

df.head()

In [ ]:
# Task 2: Write your code here:

# Here we will perform a comperhensive Missing Values investigation properly and then handle them

# General information
print("Dataset Information:")
df.info()

# Missing values per column
print("\nMissing values per column:")
print(df.isna().sum())

# Total missing values in dataset
print("\nTotal missing values in dataset:")
print(df.isna().sum().sum())

# Percentage of missing values per column
print("\nPercentage of missing values per column:")
print(df.isna().mean() * 100)

In [ ]:
# Now after we checked the Number of missing values
# We identified the Categorical Columns from Numerical Columns
# We handle Missing Values appropiatley
# Categorical Columns filled with .mode()
# Numerical Columns filled with .median()

categ_miss_cols = ["Weather","Traffic_Level","Time_of_Day"]
num_miss_cols = ["Courier_Experience_yrs",	"Delivery_Time"]

for col in categ_miss_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

for col in num_miss_cols:
    df[col] = df[col].fillna(df[col].median())

# Now we check the number of missing values after filling
print(f" Number of Missing Values in the df: {df.isna().sum().sum()}")

# GOOD!

In [ ]:
# Task 3: Write your code here:


# Here we will check the number of duplicates in the dataset
print(f"Number of Duplicates Before Cleaning: {df.duplicated().sum()}")

# Now we will Drop them And Check after that the number of duplicates
df = df.drop_duplicates()

print(f"Number of Duplicates After Cleaning: {df.duplicated().sum()}")

# Everything Looks Good!

In [ ]:
# Task 4: Write your code here: Encode categorical variables if needed (Bonus if used One Hot Encoding)

categ_cols = df.select_dtypes(include=['object']).columns

print(f"Categorical columns in the Dataset: {categ_cols}")
# After inspecting the Data Looks like We need to use LabelEncoder() Because the Ordering of the categorical column values, we do NOT USE OneHotEncoder, I JUST TRIED IT AND GOT AN ERROR!

for col in categ_cols:
    encoder = LabelEncoder()
    df[col] = encoder.fit_transform(df[col].astype(str))

# Now ew just Check the Categorical Columns in the Dataset After Encoding them

df.head()

In [ ]:
# Task 5: Write your code here: Apply feature scaling for all features (Use StandardScaler)

# Here we simply define the StandardScaler function from Sklearn and apply it to the whole dataset features
# BUT!!, Before that we need to extract all the feature names and Scale so we would be having a DataFrame at the End not a Numpy Array ! (df = sacler.fit_transform(df) -> Results into Numpy Array!)
# REMEMBER WE DONT SCALE THE TARGET AT ALL!

columns = df.columns.drop("Delivery_Time").tolist()
print(f"DataFrame Features: {columns}")

scaler = StandardScaler()

df[columns] = scaler.fit_transform(df[columns])

# We Just check the Values now
df.head()

In [ ]:
# Task 6: Write your code here: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

# WE DONT NEED TO CHECK THE CLASS IMBALANCE SINCE THE TARGET IS CONTINOUS AND NOT CATEGORICAL!!!

In [ ]:
# Task 1: Write your code here: Split the dataset into features (X) and target (y)

# Basically Assign the right Features to X and y
X = df.drop("Delivery_Time", axis=1)
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:

# Here we First Define the KFold Function from sklearn, Then we define Our Model and then Training Loop of the Folds, Finally Print out the Metrics
# ALL LIBRARIES ARE IMPORTED IN THE FIRST CELL

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for fold, (train_idx, test_idx) in enumerate(kfold.split(X), start=1):
    print(f"Fold Number: {fold}")
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train and predict
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_test, y_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  {mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:


# Here we first extract the Feature Cols (DO NOT INCLUDE THE TARGET!)
feature_cols = df.columns.drop("Delivery_Time").tolist()

# Then Group the Feature Cols with it's importance from the model respectivly and plt the Feature importance plot
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], edgecolor='black')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:

# Simply Just plot the distribution of the predicited values
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=20, edgecolor="black")
plt.title("Predicted Delivery Time Histogram")
plt.xlabel("Distribution")
plt.ylabel("Frequency")
plt.show()

# Gaussian Distribution Predictions :D

In [ ]:
# Task Bonus: Write your code here:

# First we Define Our Models
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}

# Simply Define a Dicitionary to Store the Model/Results(MAEs)

RFR_preds = []
Cat_preds = []

all_results = {}
for name in models:
  all_results[name] = {'mae': []}

# LOOP THROUGH THE MODELS
for model_name, model in models.items():
    print(f"Training {model_name}...")
    # Train
    model.fit(X_train, y_train)
    # Predict
    y_pred = model.predict(X_test)
    if model_name == "Random Forest Regressor":
        RFR_preds.append(y_pred)
    else:
        Cat_preds.append(y_pred)

# Just to access the Values
RFR_preds = RFR_preds[0]
Cat_preds = Cat_preds[0]

# Now we have collected All Models Predictions, We Basically Average them Out and Calculate the MAE
print(50 * "==")
ensemble_preds = 0.5 * RFR_preds + 0.5 * Cat_preds
print(f"Ensembled Predictions: {ensemble_preds}")
print(50 * "==")
print(f"MAE: {mean_absolute_error(y_test, ensemble_preds)}")